In [ ]:
# Cài đặt thư viện nếu chưa có
!pip install gradio -q

# Import cần thiết
import gradio as gr
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load model và tokenizer
model_path = "/flan_t5_pubmedqa_trained_full_dataset"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)

# Hàm hỏi đáp
def ask_question(question, context):
    prompt = f"Answer the medical question with yes, no or maybe.\nQuestion: {question}\nContext: {context}"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)

    outputs = model.generate(
        **inputs,
        max_length=10,
        num_beams=5,
        early_stopping=True
    )
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer

# Tạo giao diện Gradio
with gr.Blocks() as demo:
    gr.Markdown("## 🩺 Medical QA Demo (Yes/No/Maybe)")
    
    with gr.Row():
        context_input = gr.Textbox(label="Context", placeholder="Nhập ngữ cảnh y khoa...")
        question_input = gr.Textbox(label="Question", placeholder="Đặt câu hỏi...")
    
    answer_output = gr.Textbox(label="Answer")
    
    submit_btn = gr.Button("Get Answer")
    
    submit_btn.click(
        fn=ask_question,
        inputs=[question_input, context_input],
        outputs=answer_output
    )

# Chạy giao diện
demo.launch(share=True)  # share=True để có link public nếu muốn